# 03 — Milestone Forecast Validation

## Objective

In this notebook, I evaluate the reliability and internal consistency of the milestone forecasting system developed in Version 1 of my Cristiano Ronaldo 1,000-goal forecasting project.

The existing system combines a goal-scoring ensemble, an appearance prediction model, recursive match-level state updates, and Monte Carlo simulation to estimate possible paths toward Ronaldo's 1,000th career goal.

While the preceding Version 2 experiments focused on goal-model complexity, predictive performance, calibration, and uncertainty, this notebook examines the forecasting system at the simulation level.

My objectives are to:

1. Inspect the existing appearance-aware Monte Carlo forecasting implementation and its saved outputs.
2. Verify that fixture-level milestone probabilities form a mathematically consistent cumulative distribution.
3. Independently reconstruct appearance-sensitivity results from saved simulation paths.
4. Check individual simulation outcomes for inconsistencies involving milestone completion, remaining goals, and appearances.
5. Examine how changes in appearance assumptions affect milestone completion probabilities and conditional milestone dates.
6. Assess whether differences between independently generated baseline forecasts are consistent with Monte Carlo sampling variation.
7. Document the limitations of the existing forecast and determine what conclusions the available evidence supports.

## Methodology

I use the saved Version 1 forecasting outputs rather than retraining the production models or rerunning the complete simulation engine.

The analysis focuses on three components:

- **Probability integrity:** Checking fixture-level milestone probabilities, cumulative probabilities, and simulation counts.
- **Simulation consistency:** Verifying individual simulation outcomes and independently reproducing scenario-level statistics.
- **Uncertainty and sensitivity:** Examining how appearance assumptions influence milestone completion probabilities and the distribution of milestone dates.

The forecast uses a historical cutoff of **28 August 2026** and evaluates possible milestone outcomes within a projected fixture horizon ending on **28 May 2027**.

## Scope and Limitations

This notebook validates the internal consistency of saved forecasting outputs. It does not establish the historical predictive accuracy of milestone dates.

A leakage-safe historical milestone backtest would require reconstructing the forecasting system using only information available at each historical prediction date. The existing production models and saved outputs do not independently establish that requirement.

Additionally, the saved simulation paths do not expose every intermediate match-level state, limiting direct verification of the recursive simulation process.

Milestone-date quantiles are calculated only from simulation paths that reach 1,000 goals within the forecast horizon. Unfinished paths remain included when calculating completion probabilities.

I therefore distinguish between:

- **Internal consistency:** Whether the saved probabilities, simulation outcomes, and reported summaries agree.
- **Scenario sensitivity:** How the forecast changes under different appearance assumptions.
- **Predictive accuracy:** Whether the system correctly anticipates future milestone outcomes, which is not established by these checks.

## Expected Outcome

By the end of this notebook, I aim to establish whether the saved milestone forecasting outputs are internally consistent, quantify their sensitivity to appearance assumptions, and document the uncertainty and limitations associated with their interpretation.

The findings will inform whether the existing forecasting system can be retained without modification or whether further investigation is necessary.

**Version 1 remains unchanged throughout this experiment.**

In [ ]:
from pathlib import Path

project_root = Path.cwd().resolve().parents[2]

print("PROJECT ROOT")
print(project_root)

print("\nPOTENTIAL FORECASTING MODULES")
print("=" * 85)

search_terms = (
    "forecast",
    "simulat",
    "milestone",
    "appearance",
    "state_updater",
    "goal_predictor",
    "fixture_features",
)

excluded_directories = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
    "site-packages",
}

matching_files = []

for path in project_root.rglob("*.py"):

    if any(
        part in excluded_directories
        for part in path.relative_to(project_root).parts
    ):
        continue

    relative_path = path.relative_to(project_root)

    if any(
        term in path.stem.lower()
        for term in search_terms
    ):
        matching_files.append(str(relative_path))

for filename in sorted(matching_files):
    print(filename)

print(f"\nTotal matching modules: {len(matching_files)}")

In [3]:
# Step 2: Locate milestone forecasting logic

import ast
from pathlib import Path

search_keywords = (
    "milestone",
    "simulate",
    "simulation",
    "1000",
    "target_goals",
    "remaining_goals",
    "cumulative_goals",
    "poisson",
)

excluded_directories = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
    "site-packages",
}

matches = []

for path in project_root.rglob("*.py"):

    relative_path = path.relative_to(project_root)

    if any(
        part in excluded_directories
        for part in relative_path.parts
    ):
        continue

    try:
        source = path.read_text(encoding="utf-8")
        tree = ast.parse(source)

    except (UnicodeError, SyntaxError, OSError):
        continue

    for node in ast.walk(tree):

        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):

            function_name = node.name.lower()

            if any(
                keyword in function_name
                for keyword in search_keywords
            ):
                matches.append({
                    "file": str(relative_path),
                    "line": node.lineno,
                    "function": node.name,
                })

print("POTENTIAL SIMULATION FUNCTIONS")
print("=" * 100)

for match in matches:
    print(
        f"{match['file']}:{match['line']} "
        f"-> {match['function']}"
    )

print(f"\nTotal matching functions: {len(matches)}")

print("\nFILES CONTAINING MILESTONE REFERENCES")
print("=" * 100)

reference_files = []

for path in project_root.rglob("*.py"):

    relative_path = path.relative_to(project_root)

    if any(
        part in excluded_directories
        for part in relative_path.parts
    ):
        continue

    try:
        source = path.read_text(
            encoding="utf-8"
        ).lower()

    except (UnicodeError, OSError):
        continue

    if (
        "milestone" in source
        or "1000" in source
        or "simulation" in source
    ):
        reference_files.append(
            str(relative_path)
        )

for filename in sorted(reference_files):
    print(filename)

print(
    f"\nTotal files containing references: "
    f"{len(reference_files)}"
)

POTENTIAL SIMULATION FUNCTIONS
app\streamlit_app.py:700 -> load_fixture_milestone_forecast

Total matching functions: 1

FILES CONTAINING MILESTONE REFERENCES
app\streamlit_app.py
build_ronaldo_appearance_dataset.py

Total files containing references: 2


In [5]:
# Step 3: Inspect the milestone forecast loader and project structure

import ast

app_path = project_root / "app" / "streamlit_app.py"

source_lines = app_path.read_text(
    encoding="utf-8"
).splitlines()

tree = ast.parse("\n".join(source_lines))

target_function = None

for node in ast.walk(tree):
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef),
    ):
        if node.name == "load_fixture_milestone_forecast":
            target_function = node
            break

assert target_function is not None

print("MILESTONE FORECAST LOADER")
print("=" * 90)

start_line = target_function.lineno
end_line = target_function.end_lineno

for line_number in range(start_line, end_line + 1):
    print(
        f"{line_number:4d}: "
        f"{source_lines[line_number - 1]}"
    )

print("\nTOP-LEVEL PROJECT STRUCTURE")
print("=" * 90)

for path in sorted(project_root.iterdir()):
    if path.name not in {".git", ".venv", "venv"}:
        item_type = "DIR " if path.is_dir() else "FILE"
        print(f"{item_type}  {path.name}")

print("\nNOTEBOOK INVENTORY")
print("=" * 90)

excluded_directories = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
}

notebooks = []

for path in project_root.rglob("*.ipynb"):

    relative_path = path.relative_to(project_root)

    if any(
        part in excluded_directories
        for part in relative_path.parts
    ):
        continue

    notebooks.append(str(relative_path))

for notebook in sorted(notebooks):
    print(notebook)

print(f"\nTotal notebooks: {len(notebooks)}")

MILESTONE FORECAST LOADER
 700: def load_fixture_milestone_forecast():
 701:     data = pd.read_csv(
 702:         FORECAST_DIR / "fixture_milestone_probabilities.csv",
 703:         parse_dates=["date"],
 704:     )
 705: 
 706:     assert len(data) == 45, "Expected 45 forecast fixtures."
 707: 
 708:     assert data["milestone_count"].sum() == 949, (
 709:         "Milestone counts do not match the saved simulations."
 710:     )
 711: 
 712:     assert abs(
 713:         data["cumulative_milestone_probability"].iloc[-1] - 0.949
 714:     ) < 1e-9, "Final cumulative probability is incorrect."
 715: 
 716:     return data

TOP-LEVEL PROJECT STRUCTURE
DIR   .devcontainer
FILE  .gitignore
DIR   app
DIR   artifacts
DIR   assets
FILE  build_ronaldo_appearance_dataset.py
DIR   data
DIR   experiments
DIR   models
DIR   notebooks
DIR   outputs
FILE  README.md
DIR   reports
FILE  requirements.txt
DIR   src
DIR   tests

NOTEBOOK INVENTORY
experiments\v2\notebooks\01_model_complexity.ipynb
expe

In [7]:
# Step 4: Inspect the original Monte Carlo notebook structure

import json
import ast

notebook_paths = [
    project_root / "notebooks" / "09_monte_carlo_forecast.ipynb",
    project_root / "notebooks" / "09_monte_carlo_forecast_cleaned.ipynb",
]

keywords = (
    "simulation",
    "simulate",
    "poisson",
    "milestone",
    "appearance",
    "forecast",
    "1000",
    "n_sim",
)

for notebook_path in notebook_paths:

    print("\n" + "=" * 100)
    print(f"NOTEBOOK: {notebook_path.name}")
    print("=" * 100)

    with open(notebook_path, encoding="utf-8") as file:
        notebook = json.load(file)

    cells = notebook.get("cells", [])

    print(f"\nTotal cells: {len(cells)}")

    print("\nMARKDOWN HEADINGS")
    print("-" * 80)

    for index, cell in enumerate(cells):

        if cell["cell_type"] != "markdown":
            continue

        source = "".join(cell.get("source", []))

        for line in source.splitlines():

            if line.strip().startswith("#"):
                print(f"Cell {index}: {line.strip()}")

    print("\nFUNCTION DEFINITIONS")
    print("-" * 80)

    for index, cell in enumerate(cells):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in ast.walk(tree):

            if isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef),
            ):
                print(f"Cell {index}: {node.name}()")

    print("\nSIMULATION-RELATED CODE CELLS")
    print("-" * 80)

    for index, cell in enumerate(cells):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        matching_lines = [
            line.strip()
            for line in source.splitlines()
            if any(
                keyword in line.lower()
                for keyword in keywords
            )
        ]

        if matching_lines:

            print(f"\nCell {index}:")

            for line in matching_lines[:8]:
                print(f"  {line[:150]}")

            if len(matching_lines) > 8:
                print(
                    f"  ... {len(matching_lines) - 8} "
                    "additional matching lines"
                )

print("\nMonte Carlo notebook inspection completed.")


NOTEBOOK: 09_monte_carlo_forecast.ipynb

Total cells: 106

MARKDOWN HEADINGS
--------------------------------------------------------------------------------
Cell 0: # Monte Carlo Milestone Forecasting
Cell 2: ## Build the confirmed fixture calendar
Cell 4: ## Audit recursive forecast feature dependencies
Cell 7: ## Audit future-opponent historical coverage
Cell 12: ## Reconstruct the true external state at the forecast cutoff
Cell 15: ## Lock the forecast-state policy
Cell 17: ## Identify and audit all 35 Ronaldo-recursive features
Cell 25: ## Initialize the recursive simulator at the cutoff
Cell 42: ## Refit the production appearance model
Cell 46: ## Inspect the future fixture calendar
Cell 51: ## Create the recursive appearance feature builder
Cell 59: ## Run the full appearance-aware Monte Carlo

FUNCTION DEFINITIONS
--------------------------------------------------------------------------------
Cell 5: audit_feature_dependencies()
Cell 9: normalize_name()
Cell 13: points_from_r

In [9]:
# Step 5: Inspect the cleaned forecast engine

import json
import ast

cleaned_path = (
    project_root
    / "notebooks"
    / "09_monte_carlo_forecast_cleaned.ipynb"
)

with open(cleaned_path, encoding="utf-8") as file:
    cleaned_notebook = json.load(file)

cells = cleaned_notebook["cells"]

target_functions = {
    "simulate_fast_appearance_path",
    "predict_direct_ensemble_lambda",
}

print("FORECAST ENGINE FUNCTION INTERFACES")
print("=" * 100)

for index, cell in enumerate(cells):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in ast.walk(tree):

        if not isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            continue

        if node.name not in target_functions:
            continue

        print(f"\nFUNCTION: {node.name}")
        print(f"Notebook cell: {index}")
        print(f"Source lines: {len(source.splitlines())}")

        # I inspect the function signature and its dependencies.
        arguments = [
            argument.arg
            for argument in node.args.args
        ]

        print("Arguments:", arguments)

        called_functions = sorted({
            child.func.id
            for child in ast.walk(node)
            if isinstance(child, ast.Call)
            and isinstance(child.func, ast.Name)
        })

        print("\nCalled functions:")

        for name in called_functions:
            print(f"  {name}")

        print("\nReturn statements:")

        for child in ast.walk(node):

            if isinstance(child, ast.Return):
                print(
                    ast.unparse(child.value)[:500]
                    if child.value is not None
                    else "None"
                )

print("\nMONTE CARLO EXECUTION CELL")
print("=" * 100)

execution_cell = cells[68]

print(
    "".join(
        execution_cell.get("source", [])
    )[:6000]
)

print("\nForecast engine inspection completed.")

FORECAST ENGINE FUNCTION INTERFACES

FUNCTION: simulate_fast_appearance_path
Notebook cell: 56
Source lines: 131
Arguments: ['rng', 'fixtures', 'starting_state', 'starting_appearance_state', 'target_goal']

Called functions:
  build_compact_appearance_features
  deepcopy
  float
  int
  max
  predict_ensemble_lambda
  update_compact_appearance_state
  update_recursive_state

Return statements:
summary

FUNCTION: predict_direct_ensemble_lambda
Notebook cell: 62
Source lines: 110
Arguments: ['state', 'fixture']

Called functions:
  build_recursive_features
  calculate_days_rest
  count_recent_appearances
  float
  transform_xgb_direct

Return statements:
(lambda_ensemble, lambda_xgb, lambda_cat)

MONTE CARLO EXECUTION CELL
from time import perf_counter

N_SIMULATIONS = 1000  # Pilot: increase after checking runtime and output.
SEED = 2026
rng = np.random.default_rng(SEED)
start = perf_counter()
forecast_results = [
    simulate_fast_appearance_path(
        rng=rng,
        fixtures=simu

In [11]:
# Step 6: Inventory the existing forecast outputs

from pathlib import Path
import pandas as pd

forecast_dir = project_root / "outputs" / "forecast"

print("SAVED FORECAST OUTPUTS")
print("=" * 90)

if not forecast_dir.is_dir():
    print("Forecast directory not found.")
else:
    for path in sorted(forecast_dir.iterdir()):

        if not path.is_file():
            continue

        size_kb = path.stat().st_size / 1024

        print(
            f"{path.name:<48}"
            f"{size_kb:>10.1f} KB"
        )

print("\nCSV STRUCTURE")
print("=" * 90)

for path in sorted(forecast_dir.glob("*.csv")):

    try:
        df = pd.read_csv(path, nrows=5)

        print(f"\n{path.name}")
        print(f"Columns ({len(df.columns)}):")
        print(", ".join(df.columns))

    except Exception as exc:
        print(f"\n{path.name}: {type(exc).__name__}: {exc}")

print("\nForecast output inspection completed.")

SAVED FORECAST OUTPUTS
01_monthly_milestone_probability.png                 363.7 KB
02_milestone_date_distribution.png                   469.5 KB
03_appearance_sensitivity.png                        190.3 KB
appearance_sensitivity_paths.csv                     109.5 KB
appearance_sensitivity_summary.csv                     0.4 KB
fixture_milestone_probabilities.csv                    6.0 KB
future_fixtures.csv                                    5.0 KB
monthly_milestone_probabilities.csv                    1.1 KB
unfinished_goals_distribution.csv                      0.2 KB
unfinished_paths_summary.csv                           0.3 KB

CSV STRUCTURE

appearance_sensitivity_paths.csv
Columns (6):
scenario, adjustment, milestone_reached, milestone_date, goals_remaining, appearances_added

appearance_sensitivity_summary.csv
Columns (10):
scenario, simulations, completed, unfinished, completion_probability, median_date, p10_date, p90_date, mean_appearances, mean_goals_remaining

fixture_mi

In [13]:
# Step 7: Audit saved milestone forecast distributions

import numpy as np
import pandas as pd

forecast_dir = project_root / "outputs" / "forecast"

fixture_probs = pd.read_csv(
    forecast_dir / "fixture_milestone_probabilities.csv",
    parse_dates=["date"],
)

sensitivity_paths = pd.read_csv(
    forecast_dir / "appearance_sensitivity_paths.csv",
    parse_dates=["milestone_date"],
)

sensitivity_summary = pd.read_csv(
    forecast_dir / "appearance_sensitivity_summary.csv",
    parse_dates=["median_date", "p10_date", "p90_date"],
)

# I validate the fixture-level probability distribution.
fixture_probs = fixture_probs.sort_values(
    ["date", "fixture_index"]
).reset_index(drop=True)

assert fixture_probs["fixture_index"].is_unique
assert fixture_probs["date"].is_monotonic_increasing

assert fixture_probs["milestone_count"].ge(0).all()

assert fixture_probs["milestone_probability"].between(
    0, 1
).all()

assert fixture_probs[
    "cumulative_milestone_probability"
].between(0, 1).all()

assert fixture_probs[
    "cumulative_milestone_probability"
].is_monotonic_increasing

# I infer the simulation denominator from the saved probability/count pairs.
nonzero = fixture_probs[
    fixture_probs["milestone_count"] > 0
]

implied_simulations = (
    nonzero["milestone_count"]
    / nonzero["milestone_probability"]
)

assert len(nonzero) > 0

assert np.allclose(
    implied_simulations,
    implied_simulations.iloc[0],
    atol=1e-8,
)

n_simulations = int(
    round(implied_simulations.iloc[0])
)

assert n_simulations > 0

assert np.allclose(
    fixture_probs["milestone_probability"],
    fixture_probs["milestone_count"] / n_simulations,
    atol=1e-10,
)

assert np.allclose(
    fixture_probs["cumulative_milestone_probability"],
    fixture_probs["milestone_probability"].cumsum(),
    atol=1e-10,
)

# I inspect the saved appearance-sensitivity scenarios.
scenario_counts = (
    sensitivity_paths.groupby("scenario", dropna=False)
    .agg(
        simulations=("milestone_reached", "size"),
        completed=("milestone_reached", "sum"),
        unfinished=(
            "milestone_reached",
            lambda values: (~values.astype(bool)).sum()
        ),
    )
)

scenario_counts["completion_probability"] = (
    scenario_counts["completed"]
    / scenario_counts["simulations"]
)

print("FIXTURE-LEVEL FORECAST AUDIT")
print("=" * 80)

print(f"Fixtures: {len(fixture_probs)}")
print(f"Implied simulations: {n_simulations:,}")

print(
    "Milestone completions:",
    int(fixture_probs["milestone_count"].sum()),
)

print(
    "Final cumulative probability:",
    f"{fixture_probs['cumulative_milestone_probability'].iloc[-1]:.2%}",
)

print("\nAPPEARANCE-SENSITIVITY SCENARIOS")
print("=" * 80)

print(
    scenario_counts.to_string(
        float_format=lambda value: f"{value:.4f}"
    )
)

print("\nSAVED SCENARIO SUMMARY")
print("=" * 80)

print(
    sensitivity_summary[
        [
            "scenario",
            "simulations",
            "completed",
            "unfinished",
            "completion_probability",
            "median_date",
        ]
    ].to_string(index=False)
)

print("\nAll fixture probability integrity checks passed.")

FIXTURE-LEVEL FORECAST AUDIT
Fixtures: 45
Implied simulations: 1,000
Milestone completions: 949
Final cumulative probability: 94.90%

APPEARANCE-SENSITIVITY SCENARIOS
              simulations  completed  unfinished  completion_probability
scenario                                                                
Baseline             1000        942          58                  0.9420
Conservative         1000        878         122                  0.8780
Optimistic           1000        964          36                  0.9640

SAVED SCENARIO SUMMARY
    scenario  simulations  completed  unfinished  completion_probability median_date
Conservative         1000        878         122                   0.878  2027-04-02
    Baseline         1000        942          58                   0.942  2027-02-25
  Optimistic         1000        964          36                   0.964  2027-02-18

All fixture probability integrity checks passed.


In [15]:
# Step 8: Reconstruct appearance-sensitivity summaries

reconstructed = []

for scenario, group in sensitivity_paths.groupby("scenario"):

    completed = group.loc[
        group["milestone_reached"].astype(bool)
    ]

    milestone_dates = completed["milestone_date"]

    reconstructed.append({
        "scenario": scenario,
        "simulations": len(group),
        "completed": len(completed),
        "unfinished": len(group) - len(completed),
        "completion_probability": len(completed) / len(group),
        "median_date": milestone_dates.median(),
        "p10_date": milestone_dates.quantile(0.10),
        "p90_date": milestone_dates.quantile(0.90),
    })

reconstructed_df = pd.DataFrame(reconstructed)

saved = sensitivity_summary.set_index("scenario")
calculated = reconstructed_df.set_index("scenario")

print("SCENARIO SUMMARY RECONSTRUCTION")
print("=" * 85)

for scenario in calculated.index:

    print(f"\n{scenario}")
    print("-" * 60)

    for column in [
        "simulations",
        "completed",
        "unfinished",
        "completion_probability",
        "median_date",
        "p10_date",
        "p90_date",
    ]:

        actual = calculated.loc[scenario, column]
        expected = saved.loc[scenario, column]

        if column.endswith("_date"):
            matches = (
                pd.isna(actual) and pd.isna(expected)
            ) or actual == expected

        elif column == "completion_probability":
            matches = np.isclose(
                actual, expected, atol=1e-10
            )

        else:
            matches = actual == expected

        print(
            f"{column:<25}"
            f"{str(actual):<28}"
            f"{'PASS' if matches else 'MISMATCH'}"
        )

print("\nScenario reconstruction completed.")

SCENARIO SUMMARY RECONSTRUCTION

Baseline
------------------------------------------------------------
simulations              1000                        PASS
completed                942                         PASS
unfinished               58                          PASS
completion_probability   0.942                       PASS
median_date              2027-02-25 00:00:00         PASS
p10_date                 2026-12-03 12:00:00         PASS
p90_date                 2027-05-06 00:00:00         PASS

Conservative
------------------------------------------------------------
simulations              1000                        PASS
completed                878                         PASS
unfinished               122                         PASS
completion_probability   0.878                       PASS
median_date              2027-04-02 00:00:00         PASS
p10_date                 2026-12-10 00:00:00         PASS
p90_date                 2027-05-12 00:00:00         PASS

Optimisti

In [17]:
# Step 9: Validate individual simulation outcomes

paths = sensitivity_paths.copy()

paths["milestone_reached"] = paths[
    "milestone_reached"
].astype(bool)

completed = paths["milestone_reached"]
unfinished = ~completed

checks = {
    "Completed paths have milestone dates":
        paths.loc[completed, "milestone_date"].notna().all(),

    "Unfinished paths have no milestone dates":
        paths.loc[unfinished, "milestone_date"].isna().all(),

    "Completed paths have zero remaining goals":
        paths.loc[completed, "goals_remaining"].eq(0).all(),

    "Unfinished paths have positive remaining goals":
        paths.loc[unfinished, "goals_remaining"].gt(0).all(),

    "Remaining goals are nonnegative":
        paths["goals_remaining"].ge(0).all(),

    "Appearance counts are nonnegative":
        paths["appearances_added"].ge(0).all(),

    "No missing scenario labels":
        paths["scenario"].notna().all(),

    "No missing completion outcomes":
        paths["milestone_reached"].notna().all(),
}

print("INDIVIDUAL SIMULATION INTEGRITY")
print("=" * 75)

for description, passed in checks.items():
    print(
        f"{description:<58}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nUNFINISHED PATHS BY SCENARIO")
print("=" * 75)

unfinished_summary = (
    paths.loc[unfinished]
    .groupby("scenario")
    .agg(
        unfinished_paths=("goals_remaining", "size"),
        mean_goals_remaining=("goals_remaining", "mean"),
        median_goals_remaining=("goals_remaining", "median"),
        mean_appearances=("appearances_added", "mean"),
    )
)

print(
    unfinished_summary.to_string(
        float_format=lambda value: f"{value:.3f}"
    )
)

assert all(checks.values()), (
    "At least one simulation integrity check failed."
)

print("\nAll individual simulation integrity checks passed.")

INDIVIDUAL SIMULATION INTEGRITY
Completed paths have milestone dates                      PASS
Unfinished paths have no milestone dates                  PASS
Completed paths have zero remaining goals                 PASS
Unfinished paths have positive remaining goals            PASS
Remaining goals are nonnegative                           PASS
Appearance counts are nonnegative                         PASS
No missing scenario labels                                PASS
No missing completion outcomes                            PASS

UNFINISHED PATHS BY SCENARIO
              unfinished_paths  mean_goals_remaining  median_goals_remaining  mean_appearances
scenario                                                                                      
Baseline                    58                 2.862                   2.500            34.172
Conservative               122                 3.139                   2.000            31.926
Optimistic                  36                 2.250  

In [19]:
# Step 10: Compare the saved baseline Monte Carlo estimates

import numpy as np
import pandas as pd

fixture_completed = int(
    fixture_probs["milestone_count"].sum()
)

fixture_total = int(n_simulations)

baseline_paths = sensitivity_paths.loc[
    sensitivity_paths["scenario"].eq("Baseline")
]

sensitivity_completed = int(
    baseline_paths["milestone_reached"].sum()
)

sensitivity_total = len(baseline_paths)

p_fixture = fixture_completed / fixture_total
p_sensitivity = sensitivity_completed / sensitivity_total

# I estimate the Monte Carlo standard error for each proportion.
se_fixture = np.sqrt(
    p_fixture * (1 - p_fixture) / fixture_total
)

se_sensitivity = np.sqrt(
    p_sensitivity * (1 - p_sensitivity) / sensitivity_total
)

# This comparison assumes independent simulation runs.
se_difference = np.sqrt(
    se_fixture**2 + se_sensitivity**2
)

observed_difference = p_fixture - p_sensitivity

print("BASELINE MONTE CARLO COMPARISON")
print("=" * 75)

print(f"Fixture-level simulations: {fixture_total:,}")
print(f"Fixture-level completions: {fixture_completed:,}")
print(f"Fixture-level probability: {p_fixture:.2%}")
print(f"Fixture-level MC standard error: {se_fixture:.4%}")

print()

print(f"Sensitivity simulations: {sensitivity_total:,}")
print(f"Sensitivity completions: {sensitivity_completed:,}")
print(f"Sensitivity probability: {p_sensitivity:.2%}")
print(f"Sensitivity MC standard error: {se_sensitivity:.4%}")

print("\nCOMPARISON")
print("=" * 75)

print(
    f"Observed difference: "
    f"{observed_difference * 100:.2f} percentage points"
)

print(
    f"Independent-run standard error of difference: "
    f"{se_difference * 100:.2f} percentage points"
)

print(
    f"Difference / standard error: "
    f"{observed_difference / se_difference:.3f}"
)

print(
    "\nInterpretation: This diagnostic assumes independent "
    "Monte Carlo runs with comparable underlying assumptions."
)

BASELINE MONTE CARLO COMPARISON
Fixture-level simulations: 1,000
Fixture-level completions: 949
Fixture-level probability: 94.90%
Fixture-level MC standard error: 0.6957%

Sensitivity simulations: 1,000
Sensitivity completions: 942
Sensitivity probability: 94.20%
Sensitivity MC standard error: 0.7392%

COMPARISON
Observed difference: 0.70 percentage points
Independent-run standard error of difference: 1.02 percentage points
Difference / standard error: 0.690

Interpretation: This diagnostic assumes independent Monte Carlo runs with comparable underlying assumptions.


In [21]:
# Step 11: Generate the milestone forecast validation report

import json
from datetime import datetime
from pathlib import Path

report_dir = project_root / "experiments" / "v2" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "experiment": "V2 milestone forecast validation",
    "status": "validated_saved_outputs",
    "forecast_horizon_fixtures": int(len(fixture_probs)),
    "fixture_forecast": {
        "simulations": int(fixture_total),
        "completed": int(fixture_completed),
        "unfinished": int(fixture_total - fixture_completed),
        "completion_probability": float(p_fixture),
        "monte_carlo_standard_error": float(se_fixture),
    },
    "appearance_sensitivity": {},
    "baseline_comparison": {
        "fixture_completion_probability": float(p_fixture),
        "sensitivity_completion_probability": float(p_sensitivity),
        "difference_percentage_points": float(
            observed_difference * 100
        ),
        "independent_run_standard_error_percentage_points": float(
            se_difference * 100
        ),
        "standardized_difference": float(
            observed_difference / se_difference
        ),
        "assumption": (
            "Independent Monte Carlo runs with comparable "
            "underlying assumptions; not independently verified."
        ),
    },
    "validation": {
        "fixture_probability_integrity": "passed",
        "scenario_summary_reconstruction": "passed",
        "individual_simulation_integrity": "passed",
        "historical_milestone_date_backtest": "not_performed",
    },
    "limitations": [
        "Saved forecast outputs were validated without rerunning the simulation engine.",
        "Historical milestone-date predictive accuracy was not evaluated.",
        "Saved simulation paths do not expose every recursive match-level state.",
        "Milestone-date quantiles are conditional on completion within the forecast horizon.",
        "The fixture-level and sensitivity baseline experiments may use different simulation settings.",
        "Monte Carlo standard errors describe simulation sampling uncertainty, not model uncertainty.",
        "Results depend on the forecast cutoff, fixture schedule, model assumptions, and appearance scenarios.",
    ],
}

for scenario, group in sensitivity_paths.groupby("scenario"):

    completed_group = group.loc[
        group["milestone_reached"].astype(bool)
    ]

    dates = completed_group["milestone_date"]

    summary["appearance_sensitivity"][scenario] = {
        "simulations": int(len(group)),
        "completed": int(len(completed_group)),
        "unfinished": int(len(group) - len(completed_group)),
        "completion_probability": float(
            len(completed_group) / len(group)
        ),
        "conditional_median_date": (
            dates.median().isoformat()
        ),
        "conditional_p10_date": (
            dates.quantile(0.10).isoformat()
        ),
        "conditional_p90_date": (
            dates.quantile(0.90).isoformat()
        ),
    }

json_path = (
    report_dir / "milestone_forecast_validation_summary.json"
)

with json_path.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

scenario_lines = []

for scenario, values in summary[
    "appearance_sensitivity"
].items():

    median = pd.Timestamp(
        values["conditional_median_date"]
    ).strftime("%Y-%m-%d")

    scenario_lines.append(
        f"| {scenario} | "
        f"{values['completed']}/{values['simulations']} | "
        f"{values['completion_probability']:.1%} | "
        f"{median} |"
    )

markdown = f"""# V2 Milestone Forecast Validation

## Objective

I evaluated the internal consistency, probability distributions,
and appearance sensitivity of my existing appearance-aware
Monte Carlo milestone forecast.

I used saved Version 1 forecast outputs without modifying
the original forecasting system.

## Forecast Probability Integrity

- Forecast fixtures: {len(fixture_probs)}
- Monte Carlo simulations: {fixture_total:,}
- Completed paths: {fixture_completed:,}
- Unfinished paths: {fixture_total - fixture_completed:,}
- Completion probability: {p_fixture:.1%}

I verified that fixture milestone probabilities were nonnegative,
that cumulative probabilities were monotonic, and that the
cumulative distribution matched the sum of individual
fixture probabilities.

All fixture probability checks passed.

## Appearance Sensitivity

| Scenario | Completed paths | Completion probability | Conditional median date |
|---|---:|---:|---|
{chr(10).join(scenario_lines)}

I independently reconstructed the scenario summaries from
individual saved simulation paths.

All 21 scenario-summary comparisons passed.

The conservative scenario produced fewer milestone completions
and a later conditional median date than the baseline and
optimistic scenarios.

This demonstrates sensitivity to appearance assumptions,
not historical predictive accuracy.

## Individual Simulation Integrity

I verified that:

- Completed paths contain milestone dates.
- Unfinished paths have no milestone dates.
- Completed paths have zero remaining goals.
- Unfinished paths have positive remaining goals.
- Remaining goal counts are nonnegative.
- Appearance counts are nonnegative.
- Scenario labels and completion outcomes are present.

All eight integrity checks passed.

## Baseline Monte Carlo Comparison

The fixture-level forecast estimated a completion probability
of {p_fixture:.1%}, compared with {p_sensitivity:.1%} in
the separate baseline sensitivity experiment.

The observed difference was
{observed_difference * 100:.2f} percentage points.

Under an independent-run approximation, the estimated
standard error of this difference was
{se_difference * 100:.2f} percentage points.

The standardized difference was
{observed_difference / se_difference:.3f}.

The results are statistically compatible with a common
underlying probability under this approximation.

However, I did not independently establish that both
experiments used identical simulation settings.

## Interpretation of Milestone Dates

The reported milestone-date quantiles are conditional
on reaching 1,000 goals within the simulated fixture horizon.

Unfinished paths remain in the denominator when calculating
completion probabilities.

I do not interpret conditional milestone-date quantiles
as unconditional prediction intervals.

## Limitations

1. I validated saved simulation outputs rather than rerunning
   the complete recursive simulation engine.
2. I did not perform a leakage-safe historical milestone-date
   backtest.
3. The saved simulation paths do not contain every intermediate
   match-level state.
4. Monte Carlo standard errors measure simulation sampling
   uncertainty, not model uncertainty.
5. Forecast outcomes depend on the fixture schedule, forecast
   cutoff, goal model, appearance model, and scenario assumptions.
6. The two baseline experiments may differ in simulation settings.

## Conclusion

The saved milestone forecast passed the probability,
scenario-summary, and individual-path consistency checks.

The results also demonstrate meaningful sensitivity to
appearance assumptions.

These findings support the internal consistency of the
saved forecasting outputs, but they do not establish
historical milestone-date predictive accuracy.

I retain the existing Version 1 forecasting system without
promoting a new milestone forecasting model.
"""

md_path = (
    report_dir / "milestone_forecast_validation_summary.md"
)

md_path.write_text(markdown, encoding="utf-8")

print("REPORTS GENERATED")
print("=" * 75)

print(json_path.relative_to(project_root))
print(md_path.relative_to(project_root))

print("\nJSON size:", json_path.stat().st_size, "bytes")
print("Markdown size:", md_path.stat().st_size, "bytes")

print("\nNotebook 03 report generation completed.")

REPORTS GENERATED
experiments\v2\reports\milestone_forecast_validation_summary.json
experiments\v2\reports\milestone_forecast_validation_summary.md

JSON size: 2633 bytes
Markdown size: 3867 bytes

Notebook 03 report generation completed.


In [23]:
# Step 12: Final report integrity verification

import json

with json_path.open("r", encoding="utf-8") as file:
    saved_report = json.load(file)

saved_markdown = md_path.read_text(encoding="utf-8")

# I verify the saved report against the validated forecast data.
assert saved_report["forecast_horizon_fixtures"] == len(fixture_probs)

assert saved_report["fixture_forecast"]["simulations"] == fixture_total
assert saved_report["fixture_forecast"]["completed"] == fixture_completed

assert np.isclose(
    saved_report["fixture_forecast"]["completion_probability"],
    p_fixture,
)

# I verify each scenario against the underlying simulation paths.
for scenario, group in sensitivity_paths.groupby("scenario"):

    reported = saved_report["appearance_sensitivity"][scenario]

    completed_count = int(group["milestone_reached"].sum())

    assert reported["simulations"] == len(group)
    assert reported["completed"] == completed_count
    assert reported["unfinished"] == len(group) - completed_count

    assert np.isclose(
        reported["completion_probability"],
        completed_count / len(group),
    )

# I verify that the report preserves the main methodological limitations.
required_phrases = [
    "conditional",
    "historical",
    "uncertainty",
    "appearance",
    "Monte Carlo",
]

for phrase in required_phrases:
    assert phrase.lower() in saved_markdown.lower(), (
        f"Missing report concept: {phrase}"
    )

assert (
    saved_report["validation"]["historical_milestone_date_backtest"]
    == "not_performed"
)

assert saved_report["status"] == "validated_saved_outputs"

print("FINAL NOTEBOOK 03 INTEGRITY CHECK")
print("=" * 75)

print("JSON report readable: PASS")
print("Markdown report readable: PASS")
print("Fixture forecast statistics: PASS")
print("Appearance-sensitivity statistics: PASS")
print("Methodological limitations documented: PASS")
print("Historical backtest status correctly recorded: PASS")

print("\nAll final report integrity checks passed.")

FINAL NOTEBOOK 03 INTEGRITY CHECK
JSON report readable: PASS
Markdown report readable: PASS
Fixture forecast statistics: PASS
Appearance-sensitivity statistics: PASS
Methodological limitations documented: PASS
Historical backtest status correctly recorded: PASS

All final report integrity checks passed.
